# UAV Neo - Hardware Bringup / Sim-to-Real Test

Verifies that lab code developed in the simulator behaves correctly on the real
drone: coordinate frames, attitude/heading, the EKF altitude source, position,
command signs, and the velocity speed scale.

Run on the Pi with the teleop stack up:

```bash
ros2 launch uav_neo_ros2_driver teleop.launch.py
```

## SAFETY

- **Remove the propellers for every bench cell below.** Tiers 1-3a never need motors.
- **Tier 3b spins motors.** Only run it with props off, the frame restrained, the
  safety pilot armed in OFFBOARD, and the autonomy operator holding RB.
- Camera / EdgeTPU bringup is covered separately by `test_async_core_real.ipynb`.

The tiers go cheapest/safest first. Stop and fix at the first tier that fails - a
frame or sign error caught here is a one-line change, not a crash in flight.


## Tier 1 - frame math (no hardware, no ROS traffic)

Imports the actual conversion helpers from `physics_real` and checks them against
known orientations. This runs anywhere the library is importable.


In [ ]:
import sys, math
import numpy as np
import drone_core

# Put the real backend on the path so we can import its conversion helpers.
_LIB = drone_core.__file__.replace("drone_core.py", "")
sys.path.insert(1, _LIB + "real")
print("library:", _LIB)


In [ ]:
from physics_real import (
    _enu_vector_to_library,
    _enu_rates_to_library,
    _average_orientation_to_attitude,
)


def approx(a, b, tol=1e-3):
    return np.allclose(np.asarray(a, float), np.asarray(b, float), atol=tol)


def yaw_quat(deg):
    # Pure yaw about the ENU up axis -> [w, x, y, z].
    t = math.radians(deg) / 2.0
    return np.array([math.cos(t), 0.0, 0.0, math.sin(t)])


# Body ENU (forward, left, up) -> library (right, up, forward).
assert approx(_enu_vector_to_library(1, 0, 0), [0, 0, 1]), "forward -> +forward"
assert approx(_enu_vector_to_library(0, -1, 0), [1, 0, 0]), "rightward -> +right"
assert approx(_enu_vector_to_library(0, 0, 1), [0, 1, 0]), "up -> +up"

# Body ENU rates -> library (pitch, yaw, roll) rates.
assert approx(_enu_rates_to_library(1, 0, 0), [0, 0, 1]), "roll rate"
assert approx(_enu_rates_to_library(0, 1, 0), [-1, 0, 0]), "pitch rate"
assert approx(_enu_rates_to_library(0, 0, 1), [0, -1, 0]), "yaw rate"

# Attitude: heading is clockwise from north over [0, 360). N=0, E=90, S=180, W=270.
for yaw_enu, expect in [(0, 90), (90, 0), (180, 270), (270, 180)]:
    att = _average_orientation_to_attitude([yaw_quat(yaw_enu)])
    assert approx(att[2], expect), f"heading enu={yaw_enu} -> {att[2]} (want {expect})"
    assert approx(att[0], 0) and approx(att[1], 0), "level pitch/roll"

# q and -q are the same rotation; a raw component mean would cancel. This must not.
q = yaw_quat(90)
att = _average_orientation_to_attitude([q, -q, q, -q])
assert approx(att[2], 0), f"sign-flip heading corrupted: {att[2]}"

# North-boundary guard: yaw must stay strictly below 360 even for hair-negative operands.
for eps in [1e-9, 1e-12, 1e-14, 0.0]:
    att = _average_orientation_to_attitude([yaw_quat(90 + eps)])
    assert 0.0 <= att[2] < 360.0, f"guard failed at eps={eps}: {att[2]}"

print("Tier 1 PASS: vector, rate, attitude, sign-flip, and north-guard all correct")


## Tier 2 - live telemetry (disarmed, hand-carried)

Props off. Build the real drone and read every physics stream while you move the
airframe by hand. This is where the frame conversions and the EKF altitude source
are validated end to end - no arming, no motors.


In [ ]:
import time

drone = drone_core.create_drone(isSimulation=False)
drone.go_async()   # spins the ROS executor in a background thread (no Xbox START gate)
print("async executor running; waiting for topics to connect...")
time.sleep(3)


### Topic liveness

Confirms the relays the library depends on are up - including `/position`, the
relay that feeds `get_position()` and now `get_altitude()`.


In [ ]:
import subprocess

wanted = ["/mavros/imu/data", "/position", "/velocity", "/mux/cmd_vel"]
out = subprocess.run(["ros2", "topic", "list"], capture_output=True, text=True).stdout
for t in wanted:
    print(("  OK       " if t in out else "  MISSING  ") + t)


### At-rest sanity

Level and still on the bench. Expect: acceleration ~ (0, +9.8, 0) in (right, up,
forward) - gravity on the up axis; angular velocity ~ 0; pitch/roll ~ 0; heading
steady (not jumping - that would mean the attitude averaging is wrong).


In [ ]:
for _ in range(6):
    a = drone.physics.get_linear_acceleration()
    w = drone.physics.get_angular_velocity()
    att = drone.physics.get_attitude()
    print(f"accel(R,U,F)=({a[0]:+5.2f},{a[1]:+5.2f},{a[2]:+5.2f})  "
          f"gyro=({w[0]:+5.2f},{w[1]:+5.2f},{w[2]:+5.2f})  "
          f"pitch={att[0]:+6.1f} roll={att[1]:+6.1f} yaw={att[2]:6.1f}")
    time.sleep(0.5)


### Hand-carry checklist

Interrupt the kernel (stop button) to end the loop. Move the airframe slowly and
check each response:

| Motion | Expected reading |
|---|---|
| Carry **forward** | `vel(R,U,F)` forward (3rd) goes positive |
| Carry **right** | `vel` right (1st) goes positive |
| **Lift up** by hand | `alt` increases (this is the indoor-altitude fix) |
| Rotate **clockwise** (top view) | `heading` increases |
| Point at magnetic **north** | `heading` ~ 0 (the case the old Euler average corrupted) |
| Tilt **nose up** | `pitch` positive |
| Dip **right wing** | `roll` positive |
| Walk **forward** a few meters | `pos` z_north (3rd) increases, x_east (1st) stays flat |

The last column of the loop confirms `get_altitude()` tracks `get_position()` up
(`y_up`) - both come from the same rangefinder-aided EKF pose.


In [ ]:
try:
    while True:
        p = drone.physics.get_position()
        v = drone.physics.get_linear_velocity()
        att = drone.physics.get_attitude()
        alt = drone.physics.get_altitude()
        consistent = abs(alt - p[1]) < 0.05
        print(f"pos(E,U,N)=({p[0]:+5.2f},{p[1]:+5.2f},{p[2]:+5.2f})  "
              f"vel(R,U,F)=({v[0]:+5.2f},{v[1]:+5.2f},{v[2]:+5.2f})  "
              f"heading={att[2]:6.1f}  alt={alt:+5.2f}  alt==pos_up:{consistent}",
              end="\r")
        time.sleep(0.1)
except KeyboardInterrupt:
    print("\nstopped")


## Tier 3a - command signs and speed scale (disarmed)

Props off. Monitors `/mux/cmd_vel` - what the library publishes into the mux -
while commanding one axis at a time. No motors: this checks the sign and scale of
the outgoing command, independent of whether the pilot has armed.


In [ ]:
import rclpy
from geometry_msgs.msg import TwistStamped

_latest = {"msg": None}
_mon = rclpy.create_node("cmd_vel_monitor")
_mon.create_subscription(
    TwistStamped, "/mux/cmd_vel",
    lambda m: _latest.__setitem__("msg", m), 10)
rclpy.get_global_executor().add_node(_mon)
time.sleep(0.5)


def read_cmd_vel(hold=1.0):
    # flight_real republishes the last setpoint at 20 Hz, so a short hold suffices.
    time.sleep(hold)
    t = _latest["msg"].twist
    return (t.linear.x, t.linear.y, t.linear.z, t.angular.z)


print("cmd_vel monitor attached")


In [ ]:
CMD = 0.4


def axis_test(label, pitch=0, roll=0, yaw=0, throttle=0, expect=""):
    drone.flight.send_pcmd(pitch, roll, yaw, throttle)
    lx, ly, lz, az = read_cmd_vel()
    print(f"{label:22s} -> linear=({lx:+.2f},{ly:+.2f},{lz:+.2f}) "
          f"angular.z={az:+.2f}   expect: {expect}")


axis_test("pitch forward (+)", pitch=+CMD, expect="linear.x > 0")
axis_test("roll right (+)", roll=+CMD, expect="linear.y < 0  (cmd_vel is ENU left-positive)")
axis_test("yaw clockwise (+)", yaw=+CMD, expect="angular.z < 0  (cmd_vel is ENU CCW-positive)")
axis_test("throttle up (+)", throttle=+CMD, expect="linear.z > 0")
drone.flight.stop()


In [ ]:
# Programmatic sign check (same axes, asserted).
drone.flight.send_pcmd(+CMD, 0, 0, 0)
assert read_cmd_vel()[0] > 0, "pitch: linear.x should be positive"
drone.flight.send_pcmd(0, +CMD, 0, 0)
assert read_cmd_vel()[1] < 0, "roll right: linear.y should be negative"
drone.flight.send_pcmd(0, 0, +CMD, 0)
assert read_cmd_vel()[3] < 0, "yaw CW: angular.z should be negative"
drone.flight.send_pcmd(0, 0, 0, +CMD)
assert read_cmd_vel()[2] > 0, "throttle up: linear.z should be positive"
drone.flight.stop()
print("Tier 3a signs PASS")


### Speed scale

`neo_lab.send_velocity` maps a body-frame velocity in m/s to a normalized command
by dividing by `REAL_MAX_SPEED`, which must equal `mux.yaml`'s `max_speed`. A
0.3 m/s forward request should appear on `/mux/cmd_vel` as `0.3 / REAL_MAX_SPEED`,
and the mux then scales it back to 0.3 m/s.


In [ ]:
try:
    import neo_lab

    class _Shim:
        pass

    shim = _Shim()
    shim.flight = drone.flight

    neo_lab.send_velocity(shim, 0.0, 0.0, 0.3)   # (v_right, v_up, v_forward) m/s
    lx = read_cmd_vel()[0]
    expect = 0.3 / neo_lab.REAL_MAX_SPEED
    drone.flight.stop()
    print(f"send_velocity(forward=0.3) -> linear.x={lx:.3f}, expect {expect:.3f} "
          f"(REAL_MAX_SPEED={neo_lab.REAL_MAX_SPEED})")
    assert abs(lx - expect) < 1e-3, "speed scale mismatch"
    print("speed-scale PASS - confirm REAL_MAX_SPEED equals mux.yaml max_speed")
except ImportError:
    print("neo_lab is not importable from the driver repo.")
    print("Check by hand: neo_lab.REAL_MAX_SPEED must equal mux.yaml max_speed (0.5).")


## Tier 3b - motor direction (props OFF, armed)

**Props off, frame restrained.** Safety pilot armed in OFFBOARD, autonomy operator
holding RB. This is the only check for the `flight_real` roll/yaw sign fixes, which
were derived from the gamepad node rather than measured.

Each command holds for 3 seconds. Confirm the frame *tries* to:

- pitch forward: **nose-down / forward** thrust
- roll right: **right side drops** (left motors spin up)
- yaw clockwise: **nose rotates clockwise** viewed from above
- throttle up: **all motors** spin up together


In [ ]:
sweep = [
    ("pitch forward", dict(pitch=+CMD)),
    ("roll right", dict(roll=+CMD)),
    ("yaw clockwise", dict(yaw=+CMD)),
    ("throttle up", dict(throttle=+CMD)),
]
for label, kw in sweep:
    print(f">>> commanding {label} for 3 s - watch the motors")
    drone.flight.send_pcmd(kw.get("pitch", 0), kw.get("roll", 0),
                           kw.get("yaw", 0), kw.get("throttle", 0))
    time.sleep(3)
    drone.flight.stop()
    time.sleep(1)
print("Tier 3b done")


## Teardown

In [ ]:
drone.flight.stop()
print("stopped. Restart the kernel to fully release the ROS nodes.")
